In [6]:
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [7]:
df = pd.read_excel('OnlineRetail_CleanDataset.xlsx')

print("Data loaded successfully!")
print("Dataset Shape:", df.shape)

df.head()

Data loaded successfully!
Dataset Shape: (536641, 9)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,LineValue
0,574481,21791,VINTAGE HEADS AND TAILS CARD GAME,1,2011-11-04 12:45:00,1.25,18022.0,United Kingdom,1.25
1,574481,21828,EIGHT PIECE SNAKE SET,3,2011-11-04 12:45:00,1.25,18022.0,United Kingdom,3.75
2,574481,21914,BLUE HARMONICA IN BOX,1,2011-11-04 12:45:00,1.25,18022.0,United Kingdom,1.25
3,574481,21915,RED HARMONICA IN BOX,3,2011-11-04 12:45:00,1.25,18022.0,United Kingdom,3.75
4,574481,21917,SET 12 KIDS WHITE CHALK STICKS,1,2011-11-04 12:45:00,0.42,18022.0,United Kingdom,0.42


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 536641 entries, 0 to 536640
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    536641 non-null  object        
 1   StockCode    536641 non-null  object        
 2   Description  535187 non-null  object        
 3   Quantity     536641 non-null  int64         
 4   InvoiceDate  536641 non-null  datetime64[ns]
 5   UnitPrice    536641 non-null  float64       
 6   CustomerID   401604 non-null  float64       
 7   Country      536641 non-null  object        
 8   LineValue    536641 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(4)
memory usage: 36.8+ MB


In [9]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID,LineValue
count,536641.000000,536641,536641.000000,401604.000000,536641.000000
mean,9.620029,2011-07-04 08:57:06.087421952,4.632656,15281.160818,18.123861
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000,-168469.600000
25%,1.000000,2011-03-28 10:52:00,1.250000,13939.000000,3.750000
50%,3.000000,2011-07-19 14:04:00,2.080000,15145.000000,9.870000
75%,10.000000,2011-10-18 17:05:00,4.130000,16784.000000,17.400000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000,168469.600000
std,219.130156,NaN,97.233118,1714.006089,380.656263


In [11]:
df.isnull().sum()

,0
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135037
Country,0
LineValue,0


In [12]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


In [13]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df['CustomerID'] = pd.to_numeric(
    df['CustomerID'],
    errors='coerce'
)

df['LineValue'] = (
    pd.to_numeric(df['Quantity'], errors='coerce') *
    pd.to_numeric(df['UnitPrice'], errors='coerce')
)

print("Data preparation completed!")

Data preparation completed!


In [16]:
# Create RFM-specific working dataframe
rfm_df = df.copy()

# Convert required columns
rfm_df['InvoiceDate'] = pd.to_datetime(rfm_df['InvoiceDate'])
rfm_df['CustomerID'] = pd.to_numeric(
    rfm_df['CustomerID'], errors='coerce'
)

# Keep only valid customer purchases
rfm_df = rfm_df[
    rfm_df['CustomerID'].notna() &
    ~rfm_df['InvoiceNo'].astype(str).str.startswith('C') &
    (rfm_df['Quantity'] > 0) &
    (rfm_df['UnitPrice'] > 0)
].copy()

# Recalculate transaction value
rfm_df['LineValue'] = (
    rfm_df['Quantity'] * rfm_df['UnitPrice']
)

print("Original cleaned records:", len(df))
print("RFM eligible records:", len(rfm_df))
print("RFM eligible customers:", rfm_df['CustomerID'].nunique())

Original cleaned records: 536641
RFM eligible records: 392688
RFM eligible customers: 4338


In [17]:
# Reference date
reference_date = rfm_df['InvoiceDate'].max() + pd.Timedelta(days=1)

print("Latest Purchase Date:", rfm_df['InvoiceDate'].max())
print("RFM Reference Date:", reference_date)

Latest Purchase Date: 2011-12-09 12:50:00
RFM Reference Date: 2011-12-10 12:50:00


In [18]:
# Calculate RFM metrics at customer level

rfm = rfm_df.groupby('CustomerID').agg(
    Recency=('InvoiceDate',
             lambda x: (reference_date - x.max()).days),

    Frequency=('InvoiceNo', 'nunique'),

    Monetary=('LineValue', 'sum')

).reset_index()

print("RFM calculation completed!")
print("RFM Dataset Shape:", rfm.shape)

rfm.head(10)

RFM calculation completed!
RFM Dataset Shape: (4338, 4)


,CustomerID,Recency,Frequency,Monetary
0,12346.0,326,1,77183.60
1,12347.0,2,7,4310.00
2,12348.0,75,4,1797.24
3,12349.0,19,1,1757.55
4,12350.0,310,1,334.40
5,12352.0,36,8,2506.04
6,12353.0,204,1,89.00
7,12354.0,232,1,1079.40
8,12355.0,214,1,459.40
9,12356.0,23,3,2811.43


In [19]:
rfm.describe().round(2)

,CustomerID,Recency,Frequency,Monetary
count,4338.00,4338.00,4338.00,4338.00
mean,15300.41,92.54,4.27,2048.69
std,1721.81,100.01,7.70,8985.23
min,12346.00,1.00,1.00,3.75
25%,13813.25,18.00,1.00,306.48
50%,15299.50,51.00,2.00,668.57
75%,16778.75,142.00,5.00,1660.60
max,18287.00,374.00,209.00,280206.02


In [20]:
print("Total Customers:", rfm['CustomerID'].nunique())
print("Average Recency:", round(rfm['Recency'].mean(), 2))
print("Average Frequency:", round(rfm['Frequency'].mean(), 2))
print("Average Monetary:", round(rfm['Monetary'].mean(), 2))

Total Customers: 4338
Average Recency: 92.54
Average Frequency: 4.27
Average Monetary: 2048.69


In [21]:
print("Missing Values:")
print(rfm.isnull().sum())

print("\nDuplicate Customer IDs:")
print(rfm['CustomerID'].duplicated().sum())

print("\nCustomers with non-positive Monetary:")
print((rfm['Monetary'] <= 0).sum())

Missing Values:
CustomerID    0
Recency       0
Frequency     0
Monetary      0
dtype: int64

Duplicate Customer IDs:
0

Customers with non-positive Monetary:
0


In [22]:
# Recency scoring: lower recency gets a higher score
rfm['R_Score'] = pd.qcut(
    rfm['Recency'].rank(method='first'),
    q=5,
    labels=[5, 4, 3, 2, 1]
).astype(int)


# Frequency scoring: higher frequency gets a higher score
rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)


# Monetary scoring: higher monetary gets a higher score
rfm['M_Score'] = pd.qcut(
    rfm['Monetary'].rank(method='first'),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)


# Combine the three scores
rfm['RFM_Score'] = (
    rfm['R_Score'].astype(str) +
    rfm['F_Score'].astype(str) +
    rfm['M_Score'].astype(str)
)

rfm.head(10)

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,12346.0,326,1,77183.60,1,1,5,115
1,12347.0,2,7,4310.00,5,5,5,555
2,12348.0,75,4,1797.24,2,4,4,244
3,12349.0,19,1,1757.55,4,1,4,414
4,12350.0,310,1,334.40,1,1,2,112
5,12352.0,36,8,2506.04,3,5,5,355
6,12353.0,204,1,89.00,1,1,1,111
7,12354.0,232,1,1079.40,1,1,4,114
8,12355.0,214,1,459.40,1,1,2,112
9,12356.0,23,3,2811.43,4,3,5,435


In [23]:
# Check score distributions
print("Recency Score Distribution")
print(rfm['R_Score'].value_counts().sort_index())

print("\nFrequency Score Distribution")
print(rfm['F_Score'].value_counts().sort_index())

print("\nMonetary Score Distribution")
print(rfm['M_Score'].value_counts().sort_index())

Recency Score Distribution
R_Score
1    868
2    867
3    868
4    867
5    868
Name: count, dtype: int64

Frequency Score Distribution
F_Score
1    868
2    867
3    868
4    867
5    868
Name: count, dtype: int64

Monetary Score Distribution
M_Score
1    868
2    867
3    868
4    867
5    868
Name: count, dtype: int64


In [24]:
rfm[
    [
        'CustomerID',
        'Recency',
        'Frequency',
        'Monetary',
        'R_Score',
        'F_Score',
        'M_Score',
        'RFM_Score'
    ]
].head(15)

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,12346.0,326,1,77183.60,1,1,5,115
1,12347.0,2,7,4310.00,5,5,5,555
2,12348.0,75,4,1797.24,2,4,4,244
3,12349.0,19,1,1757.55,4,1,4,414
4,12350.0,310,1,334.40,1,1,2,112
5,12352.0,36,8,2506.04,3,5,5,355
6,12353.0,204,1,89.00,1,1,1,111
7,12354.0,232,1,1079.40,1,1,4,114
8,12355.0,214,1,459.40,1,1,2,112
9,12356.0,23,3,2811.43,4,3,5,435


In [27]:
def assign_segment(row):

    R = row['R_Score']
    F = row['F_Score']
    M = row['M_Score']

    if R >= 4 and F >= 4 and M >= 4:
        return 'Champions'

    elif R >= 3 and F >= 4:
        return 'Loyal Customers'

    elif R >= 4 and F >= 2:
        return 'Potential Loyalists'

    elif R >= 4 and F == 1:
        return 'New Customers'

    elif R >= 3 and F <= 2:
        return 'Promising'

    elif R <= 2 and F >= 4:
        return 'Cant Lose Them'

    elif R <= 2 and F >= 3:
        return 'At Risk'

    elif R <= 2 and F <= 2 and M >= 3:
        return 'Hibernating'

    else:
        return 'Need Attention'


rfm['Segment'] = rfm.apply(assign_segment, axis=1)

In [28]:
segment_summary = rfm.groupby('Segment').agg(
    Customer_Count=('CustomerID', 'count'),
    Total_Revenue=('Monetary', 'sum'),
    Average_Recency=('Recency', 'mean'),
    Average_Frequency=('Frequency', 'mean'),
    Average_Monetary=('Monetary', 'mean')
).reset_index()

segment_summary['Customer_Percentage'] = (
    segment_summary['Customer_Count'] /
    segment_summary['Customer_Count'].sum() * 100
)

segment_summary['Revenue_Percentage'] = (
    segment_summary['Total_Revenue'] /
    segment_summary['Total_Revenue'].sum() * 100
)

segment_summary = segment_summary.sort_values(
    by='Customer_Count', ascending=False
)

segment_summary.round(2)

,Segment,Customer_Count,Total_Revenue,Average_Recency,Average_Frequency,Average_Monetary,Customer_Percentage,Revenue_Percentage
5,Need Attention,1013,357180.96,194.67,1.28,352.60,23.35,4.02
2,Champions,942,5737952.12,12.50,11.20,6091.24,21.72,64.56
4,Loyal Customers,508,927623.75,37.21,5.12,1826.03,11.71,10.44
7,Potential Loyalists,477,496163.25,16.62,2.01,1040.17,11.00,5.58
0,At Risk,376,370556.74,162.66,2.30,985.52,8.67,4.17
8,Promising,351,158607.28,52.50,1.16,451.87,8.09,1.78
1,Cant Lose Them,285,453355.70,134.79,4.89,1590.72,6.57,5.10
3,Hibernating,249,335311.67,181.26,1.30,1346.63,5.74,3.77
6,New Customers,137,50457.42,18.08,1.00,368.30,3.16,0.57


In [29]:
total_customers = rfm['CustomerID'].nunique()

total_revenue = rfm['Monetary'].sum()

repeat_customers = (rfm['Frequency'] > 1).sum()

one_time_customers = (rfm['Frequency'] == 1).sum()

repeat_customer_rate = (
    repeat_customers / total_customers * 100
)

one_time_customer_rate = (
    one_time_customers / total_customers * 100
)

average_customer_value = rfm['Monetary'].mean()

average_orders_per_customer = rfm['Frequency'].mean()

champions_revenue_share = (
    rfm.loc[rfm['Segment'] == 'Champions', 'Monetary'].sum()
    / total_revenue * 100
)

print("Total Customers:", total_customers)
print("Total Customer Revenue: £", round(total_revenue, 2))
print("Repeat Customers:", repeat_customers)
print("One-Time Customers:", one_time_customers)
print("Repeat Customer Rate:", round(repeat_customer_rate, 2), "%")
print("One-Time Customer Rate:", round(one_time_customer_rate, 2), "%")
print("Average Customer Value: £", round(average_customer_value, 2))
print("Average Orders per Customer:", round(average_orders_per_customer, 2))
print("Champions Revenue Share:", round(champions_revenue_share, 2), "%")

Total Customers: 4338
Total Customer Revenue: £ 8887208.89
Repeat Customers: 2845
One-Time Customers: 1493
Repeat Customer Rate: 65.58 %
One-Time Customer Rate: 34.42 %
Average Customer Value: £ 2048.69
Average Orders per Customer: 4.27
Champions Revenue Share: 64.56 %


In [32]:
# Create final Power BI dataset from the master dataset
final_export = df.copy()

# Standardize CustomerID for merging
final_export['CustomerID'] = pd.to_numeric(
    final_export['CustomerID'], errors='coerce'
)

# Add a flag for records eligible for RFM analysis
final_export['RFM_Eligible'] = (
    final_export['CustomerID'].notna() &
    ~final_export['InvoiceNo'].astype(str).str.startswith('C') &
    (final_export['Quantity'] > 0) &
    (final_export['UnitPrice'] > 0)
)

# Select customer-level RFM columns
rfm_export = rfm[
    [
        'CustomerID',
        'Recency',
        'Frequency',
        'Monetary',
        'R_Score',
        'F_Score',
        'M_Score',
        'RFM_Score',
        'Segment'
    ]
].copy()

# Merge RFM information into the master transaction dataset
final_export = final_export.merge(
    rfm_export,
    on='CustomerID',
    how='left',
    validate='many_to_one'
)

# Export one combined dataset
final_export.to_csv(
    'Customer_Segmentation_RFM.csv',
    index=False
)

print("Final Dataset Shape:", final_export.shape)
print("Total Records:", len(final_export))
print("Unique Customers:", final_export['CustomerID'].nunique())
print("RFM Eligible Records:", final_export['RFM_Eligible'].sum())
print("Export completed successfully!")

Final Dataset Shape: (536641, 18)
Total Records: 536641
Unique Customers: 4372
RFM Eligible Records: 392688
Export completed successfully!
